# Week 6 — Conditional Generation with Classifier-Free Guidance

Every model up through Week 5 just generated *a* digit, picked by whatever the noise happened to collapse into. This week I want to ask for a specific digit and get it. The mechanism is surprisingly simple: feed the class label into the UNet the same way I feed in the timestep, and occasionally hide the label during training so the model also learns what "no label" looks like. That second part is what makes classifier-free guidance possible later.

In [ ]:
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.manual_seed(0)

In [ ]:
class NoiseScheduler:
    def __init__(self, timesteps=1000, s=0.008, device=device):
        self.timesteps = timesteps
        steps = torch.arange(timesteps + 1, dtype=torch.float64) / timesteps
        f_t = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
        alphas_cumprod = f_t / f_t[0]
        alphas_cumprod = torch.clamp(alphas_cumprod, min=1e-9)

        self.alphas_cumprod = alphas_cumprod[1:].float().to(device)
        alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), self.alphas_cumprod[:-1]])
        self.alphas_cumprod_prev = alphas_cumprod_prev.to(device)
        self.betas = (1 - self.alphas_cumprod / self.alphas_cumprod_prev).clamp(max=0.999)
        self.alphas = 1.0 - self.betas

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.alphas_cumprod[t].sqrt().view(-1, 1, 1, 1)
        sqrt_one_minus_ac = (1 - self.alphas_cumprod[t]).sqrt().view(-1, 1, 1, 1)
        return sqrt_ac * x0 + sqrt_one_minus_ac * noise, noise

scheduler = NoiseScheduler(timesteps=1000, device=device)

## Adding a label embedding

I'm reusing the Week 4/5 UNet almost unchanged. The only architectural addition is a label embedding table with `num_classes + 1` rows — the extra row is a dedicated "null" token used when a sample's label gets dropped during training, or when I deliberately want the unconditional prediction at sampling time. The label embedding gets added to the timestep embedding before either is injected into the ResBlocks, so from the ResBlock's point of view there's just a single combined conditioning vector.

In [ ]:
class SinusoidalTimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.cond_proj = nn.Linear(cond_dim, out_ch)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.cond_proj(cond_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.block = ResBlock(in_ch, out_ch, cond_dim)
        self.pool = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)

    def forward(self, x, cond_emb):
        h = self.block(x, cond_emb)
        return self.pool(h), h


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, cond_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.block = ResBlock(in_ch + skip_ch, out_ch, cond_dim)

    def forward(self, x, skip, cond_emb):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.pad(x, (0, skip.shape[-1] - x.shape[-1], 0, skip.shape[-2] - x.shape[-2]))
        x = torch.cat([x, skip], dim=1)
        return self.block(x, cond_emb)


class ConditionalUNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64, time_dim=128, num_classes=10):
        super().__init__()
        self.time_embed = SinusoidalTimestepEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, time_dim * 4), nn.SiLU(), nn.Linear(time_dim * 4, time_dim))

        # +1 row reserved for the null/no-label token used during CFG training and unconditional sampling
        self.null_label = num_classes
        self.label_embed = nn.Embedding(num_classes + 1, time_dim)

        self.inc = ResBlock(in_ch, base_ch, time_dim)
        self.down1 = Down(base_ch, base_ch * 2, time_dim)
        self.down2 = Down(base_ch * 2, base_ch * 4, time_dim)
        self.down3 = Down(base_ch * 4, base_ch * 8, time_dim)
        self.bottleneck = ResBlock(base_ch * 8, base_ch * 8, time_dim)
        self.up1 = Up(base_ch * 8, base_ch * 8, base_ch * 4, time_dim)
        self.up2 = Up(base_ch * 4, base_ch * 4, base_ch * 2, time_dim)
        # Down1's skip tensor has base_ch*2 channels (the ResBlock's out_ch, not in_ch) — skip_ch must match that.
        self.up3 = Up(base_ch * 2, base_ch * 2, base_ch, time_dim)
        self.outc = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x, t, labels):
        cond_emb = self.time_mlp(self.time_embed(t)) + self.label_embed(labels)

        h0 = self.inc(x, cond_emb)
        h1, skip1 = self.down1(h0, cond_emb)
        h2, skip2 = self.down2(h1, cond_emb)
        h3, skip3 = self.down3(h2, cond_emb)
        h3 = self.bottleneck(h3, cond_emb)
        h = self.up1(h3, skip3, cond_emb)
        h = self.up2(h, skip2, cond_emb)
        h = self.up3(h, skip1, cond_emb)
        return self.outc(h)

model = ConditionalUNet().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
NUM_CLASSES = 10
NULL_LABEL = NUM_CLASSES

## Training with conditioning dropout

At training time, for each sample I flip a coin (10% probability here) and replace the real label with the null token. This forces the model to also learn $\epsilon_\theta(x_t, t, \varnothing)$ — the unconditional noise prediction — using the exact same network. Without this, there'd be no way to compute the unconditional term that classifier-free guidance needs at sampling time, short of training a second model.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
epochs = 25
cond_dropout_prob = 0.1
losses = []

for epoch in range(epochs):
    running_loss = 0.0
    for x0, labels in train_loader:
        x0 = x0.to(device)
        labels = labels.to(device)

        drop_mask = torch.rand(labels.shape[0], device=device) < cond_dropout_prob
        train_labels = torch.where(drop_mask, torch.full_like(labels, NULL_LABEL), labels)

        t = torch.randint(0, scheduler.timesteps, (x0.shape[0],), device=device)
        xt, noise = scheduler.add_noise(x0, t)
        pred_noise = model(xt, t, train_labels)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item() * x0.shape[0]

    epoch_loss = running_loss / len(train_dataset)
    losses.append(epoch_loss)
    print(f"epoch {epoch+1}/{epochs}  loss={epoch_loss:.4f}")

plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training loss")
plt.show()

## Classifier-free guidance sampling

At each reverse step I run the model twice on the same `x_t`: once with the real class label, once with the null label, then combine:

$$\epsilon_{\text{guided}} = \epsilon_{\text{uncond}} + w \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$$

where `w` is the guidance scale. At `w=0` this is just the unconditional prediction — the label is ignored entirely. At `w=1` it's exactly the plain conditional prediction, no extrapolation. Pushing `w` above 1 extrapolates *past* the conditional prediction in the direction away from the unconditional one, which in practice sharpens class fidelity at the cost of diversity and, if pushed too far, sample quality (oversaturated, artifact-y images).

I'm using the DDIM sampler from Week 5 here since it's faster and deterministic, which makes the guidance-scale comparison easier to read (same starting noise across scales).

In [ ]:
@torch.no_grad()
def cfg_ddim_sample(model, scheduler, labels, num_steps=50, guidance_scale=3.0, device=device, x_T=None):
    model.eval()
    T = scheduler.timesteps
    shape = (labels.shape[0], 1, 28, 28)

    step_indices = torch.linspace(0, T - 1, num_steps).long().flip(0).to(device)
    x = torch.randn(shape, device=device) if x_T is None else x_T
    null_labels = torch.full_like(labels, NULL_LABEL)
    ac = scheduler.alphas_cumprod

    for i in range(len(step_indices)):
        t = step_indices[i]
        t_batch = torch.full((shape[0],), t.item(), device=device, dtype=torch.long)

        ac_t = ac[t]
        ac_prev = ac[step_indices[i + 1]] if i + 1 < len(step_indices) else torch.tensor(1.0, device=device)

        eps_cond = model(x, t_batch, labels)
        eps_uncond = model(x, t_batch, null_labels)
        eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        x0_pred = ((x - (1 - ac_t).sqrt() * eps) / ac_t.sqrt()).clamp(-1, 1)
        dir_coeff = torch.sqrt((1 - ac_prev).clamp(min=0))
        x = ac_prev.sqrt() * x0_pred + dir_coeff * eps

    model.train()
    return x.clamp(-1, 1)

## A full 0-9 grid

One column per digit class, several samples per column, all generated with a fixed moderate guidance scale.

In [ ]:
samples_per_class = 4
labels = torch.arange(NUM_CLASSES, device=device).repeat_interleave(samples_per_class)

samples = cfg_ddim_sample(model, scheduler, labels, num_steps=50, guidance_scale=3.0)

grid = make_grid(samples, nrow=NUM_CLASSES, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(12, 5))
plt.imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
plt.axis("off")
plt.title("One column per class (0-9), guidance_scale=3.0")
plt.show()

## Guidance scale sweep

Same starting noise, same target class, varying only `guidance_scale`. I expect low scales to sometimes drift toward a different digit (since the unconditional prediction dominates), and very high scales to look harsh/oversaturated even if clearly the right digit.

In [ ]:
target_class = 7
scales = [0.0, 1.0, 3.0, 6.0, 10.0]
n_samples = 4

torch.manual_seed(123)
x_T = torch.randn((n_samples, 1, 28, 28), device=device)
labels = torch.full((n_samples,), target_class, device=device)

fig, axes = plt.subplots(1, len(scales), figsize=(5 * len(scales), 5))
for ax, scale in zip(axes, scales):
    samples = cfg_ddim_sample(model, scheduler, labels, num_steps=50, guidance_scale=scale, x_T=x_T.clone())
    grid = make_grid(samples, nrow=2, normalize=True, value_range=(-1, 1))
    ax.imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    ax.set_title(f"w = {scale}")
    ax.axis("off")
plt.suptitle(f"Guidance scale sweep, target class = {target_class}")
plt.tight_layout()
plt.show()

## Self-check questions

**1. Why do we need to train with the label sometimes dropped, instead of always providing it?**

If the model only ever sees real labels during training, it never learns what the unconditional noise prediction $\epsilon_\theta(x_t, t, \varnothing)$ looks like. Classifier-free guidance needs both the conditional and unconditional prediction from the *same* network to compute the extrapolated direction. Dropping the label some fraction of the time (10% here) teaches a single model to do both jobs, which is what makes guidance possible without training a separate unconditional model from scratch.

**2. Walk through the classifier-free guidance formula. What happens at `guidance_scale = 0`? At `guidance_scale = 1`? As it grows large?**

The formula is $\epsilon_{\text{guided}} = \epsilon_{\text{uncond}} + w(\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$. At `w=0` the second term vanishes and you just get $\epsilon_{\text{uncond}}$ — the label is completely ignored and you're back to unconditional generation. At `w=1` the two unconditional terms cancel out the offset and you get exactly $\epsilon_{\text{cond}}$, the plain conditional prediction with no extrapolation. As `w` grows past 1, you're extrapolating beyond the conditional prediction in the direction that the label pushes it relative to the unconditional baseline — pushing the output further toward whatever made that class distinctive, at the cost of looking less like a typical, diverse sample from that class.

**3. How is the label embedding combined with the timestep embedding inside a ResBlock?**

They're combined before the ResBlock ever sees them — I add the label embedding vector directly to the (MLP-projected) timestep embedding vector, producing one combined conditioning vector of the same dimensionality. That single vector is what gets passed into every ResBlock and projected per-block to the right channel count before being broadcast-added to the feature map. The ResBlocks themselves have no idea whether they're seeing time information, label information, or both — they just see "the conditioning vector."

**4. What's the tradeoff as you increase the guidance scale too far?**

Sample fidelity to the class goes up — digits look more unambiguously like the target class — but diversity within that class collapses and texture quality degrades. The sweep above shows this directly: at `w=10` the digit is unmistakably a 7, but it starts to look harsh and over-contrasted compared to the more naturally-varied output at `w=3`. There's a real ceiling past which you're trading away realism for an even more pronounced label signal you didn't need.